# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a step-by-step guide for exploring the FAIR^2 clinical dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the FAIR^2 dataset Croissant schema URL:
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()

print(f"{metadata_json['name']}\n\n{metadata_json['description']}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. All entities are referenced by their `@id`. This allows clear discovery and referencing.

In [ ]:
# Show all available record sets and their fields
record_sets = dataset.record_sets()
print("Record sets in this dataset:")
for rs in record_sets:
    print(f"RecordSet Name: {rs['name']}")
    print(f"RecordSet @id: {rs['@id']}")
    print("Fields:")
    for field in rs.get('fields', []):
        print(f"    Field Name: {field['name']} | Field @id: {field['@id']}")
    print("---")

## 3. Data Extraction
Load table data from each record set into a DataFrame for detailed analysis. Use only the record set and field `@id`s discovered above.

In [ ]:
# Extract data from each record set
# Dynamically collect record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]  # All record set @ids
dataframes = {}

for record_set_id in record_set_ids:
    # Retrieve records from this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    print(f"\nExtracted DataFrame for RecordSet @id: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head())
    dataframes[record_set_id] = df

# For illustration, select the first available record set id
main_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps—filtering, normalization, and grouping to prepare data for further analysis. All references use `@id` for consistency.

In [ ]:
from numpy import nan

# For demonstration, pick a numeric field and group field dynamically
df = dataframes[main_record_set_id]
# Find numeric (float or int) columns
numeric_fields = [col for col in df.columns if (df[col].dtype == 'float64' or df[col].dtype == 'int64')]

if numeric_fields:
    numeric_field_id = numeric_fields[0]  # Reference by @id
else:
    numeric_field_id = df.columns[0]  # fallback

# Try to find a categorical/group field
group_fields = [col for col in df.columns if df[col].dtype == 'object']
group_field_id = group_fields[0] if group_fields else numeric_field_id

threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    
    # Normalization
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution
if numeric_field_id in df.columns:
    plt.figure(figsize=(7,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Visualize relationship between numeric and group field
if numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(7,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to explore the FAIR^2 clinical dataset using `mlcroissant`, referencing entities by their `@id`. You reviewed record sets and fields, extracted and processed records, and visualized relationships in the data.

- All dataset entities are referenced by their `@id`, ensuring clarity and reproducibility.
- Data exploration steps including normalization, filtering, and grouping are shown for easy adaptation to your domain.
- For further analysis, consider deeper domain-specific operations or additional visualizations, always referencing via `@id`.

**Note**: Due to the small sample size and the rare-disease context, all processing steps should be interpreted with domain knowledge.